# 009 — Broadcast weak labels → tactical boosting D/LIVE

Отдельный HistGradientBoostingClassifier: те же 92 тактических признака и параметры, что в 008, новая слабая разметка из синхронизации broadcast. Broadcast, события и часы матча не входят в признаки. H/A не предсказываются.

Prerequisites / Подготовка: запустить подготовку признаков 008 (до обучения) и локальный `local_review/broadcast_alignment/alignment.ipynb`. Затем Run All этого ноутбука. Требуются features.npy, feature_names.json из 008 и training_weak_D_mask.npy / training_weak_LIVE_mask.npy из синхронизации. Все эти данные остаются локальными; путь к кэшу явно проверяется. tqdm и progress.json показывают ход обучения.

Публичное окно 11185:18685 и по 20 секунд с обеих сторон исключены. До 5 FPS, до 50 точек на непрерывный фрагмент; равный вклад фрагментов, затем классов. Проверка: первый тайм → второй на слабых метках, без выбора гиперпараметров. Финальная модель обучается на обоих таймах вне публичного окна. Истинный публичный эталон открывается после предсказания. Ранее исследованное публичное окно — диагностика, не независимый тест.

150 деревьев, глубина 3, 7 листьев, learning_rate 0.05, min_samples_leaf 25, L2 2, seed 42, early stopping отключён. Порог 0.5. Offline-фильтр заполняет LIVE-разрывы ≤10 кадров между D и удаляет D короче 25 кадров, отдельно по таймам. Признаки используют прошлые окна до 10 секунд; слабые метки и фильтрация используют будущий контекст. Гиперпараметры не подбираются по публичным результатам.

Слабые метки не являются истиной; модель может наследовать ошибки геометрического отбора. Результаты, признаки, прогнозы и исполненный ноутбук — local_review/experiments/009_broadcast_weak_boosting/. model.pkl — доверенный собственный pickle, не загружать чужие pickle. Бинарный CSV не является конкурсной заявкой. spellF1 и H/A/D macroF1 неприменимы до добавления H/A.


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS","4")
from pathlib import Path
import sys, json, time, warnings, hashlib, itertools, importlib.metadata
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "games.py").exists())
sys.path.insert(0,str(ROOT))
from games import GAME, GAMES, asset_paths, half_video_blocks
from starter.baseline import team_of_each_track
from starter.load_smart_tagging import load as load_smart_tagging
from evaluation import possession_metrics as pm
EXPERIMENT = "009_broadcast_weak_boosting"
OUT = ROOT / "local_review/experiments" / EXPERIMENT
OUT.mkdir(parents=True,exist_ok=True)
ignore = ROOT / "local_review/.gitignore"
if not ignore.exists(): ignore.write_text("*\n")
CFG = json.loads((ROOT / "sol/001_box5_vote100/config.json").read_text())

N, FPS = GAMES[GAME]["n_video_frames"], GAMES[GAME]["fps"]
PITCH = np.array(GAMES[GAME]["pitch"],dtype=np.float64)
BLOCKS = half_video_blocks(GAME)
PATHS = asset_paths()

def save_json(path, value):
    temp = path.with_suffix(path.suffix+".tmp")
    temp.write_text(json.dumps(value,indent=2,ensure_ascii=False))
    temp.replace(path)

def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

started=time.perf_counter()
print(EXPERIMENT, "| Results:", OUT)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_recall_fscore_support,average_precision_score,confusion_matrix
import pickle
PUBLIC_START,PUBLIC_END=11185,18685
GUARD=20*FPS
PARAMS=dict(max_iter=150,learning_rate=.05,max_leaf_nodes=7,max_depth=3,
            min_samples_leaf=25,l2_regularization=2.,early_stopping=False,random_state=42)


In [ ]:
# Reuse identical tactical features to isolate the supervision change.
# Используем те же тактические признаки, чтобы сравнить именно новую разметку.
cache=ROOT/'local_review/experiments/008_weak_dead_boosting'
alignment=ROOT/'local_review/broadcast_alignment'
required=[cache/'features.npy',cache/'feature_names.json',alignment/'training_weak_D_mask.npy',alignment/'training_weak_LIVE_mask.npy']
assert all(p.exists() for p in required), 'Run feature preparation 008 and alignment.ipynb first / Сначала подготовьте признаки и синхронизацию'
X=np.load(cache/'features.npy',mmap_mode='r')
feature_names=json.loads((cache/'feature_names.json').read_text())
assert X.shape==(N,len(feature_names))
positive=np.load(alignment/'training_weak_D_mask.npy')
negative=np.load(alignment/'training_weak_LIVE_mask.npy')
assert not (positive&negative).any()
graded=np.zeros(N,bool)
for _,a,b in BLOCKS:graded[a:b+1]=True
allowed=graded.copy();allowed[PUBLIC_START-GUARD:PUBLIC_END+GUARD]=False
assert not ((positive|negative)&~allowed).any()
weak_y=np.full(N,-1,dtype=np.int8);weak_y[negative]=0;weak_y[positive]=1
indices=[];weights=[];group_ids=[];group=0
# Limit correlated frames and equalise episode contribution within each class.
# Ограничиваем коррелированные кадры и выравниваем вклад фрагментов внутри класса.
for label in [0,1]:
 mask=weak_y==label
 starts=np.flatnonzero(mask&np.r_[True,~mask[:-1]])
 stops=np.flatnonzero(mask&np.r_[~mask[1:],True])+1
 for a,b in zip(starts,stops):
  chosen=np.arange(a,b,5)
  if len(chosen)>50:chosen=chosen[np.linspace(0,len(chosen)-1,50).astype(int)]
  indices.extend(chosen.tolist());weights.extend([1/len(chosen)]*len(chosen));group_ids.extend([group]*len(chosen));group+=1
indices=np.array(indices);weights=np.array(weights);group_ids=np.array(group_ids)
assert not np.any((indices>=PUBLIC_START-GUARD)&(indices<PUBLIC_END+GUARD))
selection_summary={str(label):dict(frames=int((weak_y==label).sum()),samples=int((weak_y[indices]==label).sum()),fragments=int(len(np.unique(group_ids[weak_y[indices]==label])))) for label in [0,1]}
save_json(OUT/'weak_label_summary.json',selection_summary)
save_json(OUT/'input_hashes.json',{str(p.relative_to(ROOT)):sha256(p) for p in required})
np.savez(OUT/'weak_training_selection.npz',indices=indices,labels=weak_y[indices],group_ids=group_ids,weights=weights)
print(selection_summary)


In [ ]:
def balanced_weights(ids):
    w=weights[ids].copy();target=weak_y[indices[ids]]
    # Equal total class influence AFTER equalising run weights.
    # Выравниваем вклад классов ПОСЛЕ выравнивания весов серий.
    for label in [0,1]:
        mask=target==label;assert mask.any()
        w[mask]/=w[mask].sum()
    return w*len(w)/w.sum()

def binary_scores(truth,pred,prob=None):
    precision,recall,f1,support=precision_recall_fscore_support(truth,pred,labels=[0,1],zero_division=0)
    r=dict(D_precision=float(precision[1]),D_recall=float(recall[1]),D_F1=float(f1[1]),
           LIVE_F1=float(f1[0]),binary_macroF1=float(f1.mean()),true_D_frames=int(support[1]),predicted_D_frames=int(np.sum(pred==1)))
    if prob is not None:r['D_average_precision']=float(average_precision_score(truth,prob))
    return r

def fit_model(ids,stage):
    args=PARAMS.copy();args.pop('max_iter')
    model=HistGradientBoostingClassifier(**args,warm_start=True,max_iter=25)
    for trees in tqdm(range(25,PARAMS['max_iter']+1,25),desc=stage):
        model.set_params(max_iter=trees)
        model.fit(X[indices[ids]],weak_y[indices[ids]],sample_weight=balanced_weights(ids))
        save_json(OUT/'progress.json',dict(stage=stage,trees=trees,total_trees=PARAMS['max_iter']))
    return model
# A temporal weak-label check: first half trains, second half validates.
# Временная проверка слабых меток: обучение на первом тайме, проверка на втором.
half2=BLOCKS[1][1]
train_ids=np.flatnonzero(indices<half2);validation_ids=np.flatnonzero(indices>=half2)
assert set(group_ids[train_ids]).isdisjoint(set(group_ids[validation_ids]))
diagnostic=fit_model(train_ids,'Weak temporal check / Проверка слабых меток')
p=diagnostic.predict_proba(X[indices[validation_ids]])[:,1]
save_json(OUT/'weak_validation_metrics.json',binary_scores(weak_y[indices[validation_ids]],p>=.5,p))
# Fixed parameters; weak validation is not used to search thresholds or select trees.
# Параметры фиксированы; по слабой проверке не выбираем порог или число деревьев.
model=fit_model(np.arange(len(indices)),'Final boosting / Итоговый бустинг')
with (OUT/'model.pkl').open('wb') as f:pickle.dump(model,f)
save_json(OUT/'model_config.json',dict(parameters=PARAMS,feature_names=feature_names,
    threshold=.5,min_dead_frames=25,max_live_gap_frames=10,
    public_exclusion=[PUBLIC_START-GUARD,PUBLIC_END+GUARD],
    events_as_features=False,public_labels_used_for_training=False,
    versions={k:importlib.metadata.version(k) for k in ['numpy','pandas','scikit-learn']}))
probability=np.full(N,np.nan,dtype=np.float32)
for a in tqdm(range(0,N,10000),desc='Full D probability / Прогноз D'):
    b=min(a+10000,N);probability[a:b]=model.predict_proba(X[a:b])[:,1]
np.save(OUT/'p_D.npy',probability)


In [ ]:
def spans(values):
    if not len(values):return []
    start=np.r_[0,np.flatnonzero(values[1:]!=values[:-1])+1]
    return list(zip(start,np.r_[start[1:],len(values)]))
def duration_filter(raw):
    joined=raw.copy()
    for a,b in spans(raw):
        if not raw[a] and a>0 and b<len(raw) and b-a<=10:joined[a:b]=True
    result=joined.copy()
    for a,b in spans(joined):
        if joined[a] and b-a<25:result[a:b]=False
    return result
assert not duration_filter(np.r_[np.zeros(30,bool),np.ones(10,bool),np.zeros(30,bool)]).any()
assert duration_filter(np.r_[np.ones(25,bool),np.zeros(10,bool),np.ones(25,bool)]).all()
raw_dead=(probability>=.5)&graded
filtered=np.zeros(N,bool)
for _,first,last in BLOCKS:
    filtered[first:last+1]=duration_filter(raw_dead[first:last+1])
    assert all(b-a>=25 for a,b in spans(filtered[first:last+1]) if filtered[first+a])
labels=np.where(graded,np.where(filtered,'D','LIVE'),'X')
pd.DataFrame({'frame':np.arange(N),'p_D':np.where(graded,probability,np.nan),'label':labels}).to_csv(OUT/'binary_predictions.csv',index=False)
np.save(OUT/'dead_mask.npy',filtered)
# Ground truth is opened ONLY AFTER all fitting and probability predictions are finished.
# Эталон открываем ТОЛЬКО ПОСЛЕ обучения и получения вероятностей.
gt=pd.read_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')
assert np.array_equal(gt.frame.to_numpy(),np.arange(N))
public=gt.label.isin(['H','A','D']).to_numpy();assert public.sum()==7500
assert not public[indices].any()
target=gt.label.eq('D').to_numpy()
# Audit full weak candidates separately from the deliberately excluded training masks.
# Полные слабые кандидаты проверяем отдельно от масок обучения с исключённым публичным окном.
audit_y=np.full(N,-1,dtype=np.int8)
audit_y[np.load(alignment/'weak_LIVE_candidates.npy')]=0
audit_y[np.load(alignment/'weak_D_candidates.npy')]=1
report={}
for name,mask in [('public_5min',public),('last_minute',public&(np.arange(N)>=PUBLIC_START+6000))]:
    report[name]={
      'boosting_raw':binary_scores(target[mask],raw_dead[mask],probability[mask]),
      'boosting_filtered':binary_scores(target[mask],filtered[mask]),
      'weak_rules_as_detector_unknown_is_LIVE':binary_scores(target[mask],np.load(alignment/'weak_D_candidates.npy')[mask]),
      'weak_label_coverage':float(np.mean(audit_y[mask]>=0)),
      'true_D_runs':sum(target[mask][a] for a,b in spans(target[mask])),
      'predicted_D_runs':sum(filtered[mask][a] for a,b in spans(filtered[mask]))}
    # JSON native integers / Обычные целые числа для JSON
    report[name]['true_D_runs']=int(report[name]['true_D_runs'])
    report[name]['predicted_D_runs']=int(report[name]['predicted_D_runs'])
report['weak_public_audit']=dict(
    true_D_with_positive_weak_label=int((public&target&(audit_y==1)).sum()),
    true_D_with_negative_weak_label=int((public&target&(audit_y==0)).sum()),
    true_D_without_weak_label=int((public&target&(audit_y==-1)).sum()),
    true_LIVE_with_positive_weak_label=int((public&~target&(audit_y==1)).sum()),
    true_D_ball_missing=int((public&target&(X[:,feature_names.index('ball_seen')]==0)).sum()))
save_json(OUT/'metrics.json',report)
lines=['# 009 — бустинг D/LIVE на broadcast слабой разметке','',
'Модель не обучалась на публичных 5 минутах и в защитном окне 20 секунд вокруг них. Однако этот матч и публичные метрики ранее исследовались: это диагностика, не новый независимый тест.',
'Вход: геометрия и движение. Broadcast используется только для слабых обучающих меток. Проверка на слабых метках не является оценкой истинного качества.',
'', '| Метод, публичные 5 минут | D precision | D recall | D F1 | LIVE F1 | binary macroF1 |','|---|---:|---:|---:|---:|---:|']
for name,r in report['public_5min'].items():
    if isinstance(r,dict):lines.append(f"| {name} | {r['D_precision']:.4f} | {r['D_recall']:.4f} | {r['D_F1']:.4f} | {r['LIVE_F1']:.4f} | {r['binary_macroF1']:.4f} |")
lines+=['','Правила как детектор: неизвестные слабые метки трактуются как LIVE только для этой контрольной оценки. При обучении неизвестные исключены.',
'Сглаживание: заполнение LIVE-разрывов до 10 кадров между D, удаление D короче 25 кадров. Параметры не подбирались по публичному эталону.',
'Это только бинарная модель: H/A не назначены. spellF1 и трёхклассовый macroF1 для неё не рассчитываются.',
'Все данные и реконструируемые признаки остаются локальными.']
(OUT/'REVIEW_RU.md').write_text('\n'.join(lines)+'\n')
save_json(OUT/'progress.json',dict(status='completed',features=len(feature_names),seconds=time.perf_counter()-started,
    model_sha256=sha256(OUT/'model.pkl'),prediction_sha256=sha256(OUT/'binary_predictions.csv')))
print(json.dumps(report,indent=2))


In [ ]:
# Verify trusted model round-trip and the output contract.
# Проверяем повторную загрузку своей модели и контракт бинарных прогнозов.
with (OUT/'model.pkl').open('rb') as f:restored=pickle.load(f)
check=np.arange(0,N,251)
assert np.allclose(restored.predict_proba(X[check])[:,1],probability[check],atol=1e-7)
assert len(labels)==199319 and set(labels)<= {'D','LIVE','X'}
assert np.isfinite(probability).all()
assert all(b-a>=25 for a,b in spans(filtered) if filtered[a])
config=json.loads((OUT/'model_config.json').read_text())
config.update(supervision='broadcast geometry-aligned weak masks',broadcast_as_features=False,
              inference_context='causal tactical features; offline duration filtering',input_hashes=json.loads((OUT/'input_hashes.json').read_text()))
save_json(OUT/'model_config.json',config)
print('Checks passed / Проверки пройдены')
